# Final Test Results

In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda3/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("best_hp_reg.json") as f:
    hp = json.load(f)
print(f"Entries: {len(hp)}")  # should be 21
print(sorted(hp.keys()))

Entries: 21
['gatv2_pbsg', 'gatv2_smiles', 'gatv2_smiles_gl', 'gcn_pbsg', 'gcn_smiles', 'gcn_smiles_gl', 'gine_pbsg', 'gine_smiles', 'gine_smiles_gl', 'rf_fp_poly', 'rf_fp_pooled_poly', 'rf_fp_pooled_ru', 'rf_fp_ru', 'rr_fp_poly', 'rr_fp_pooled_poly', 'rr_fp_pooled_ru', 'rr_fp_ru', 'xgb_fp_poly', 'xgb_fp_pooled_poly', 'xgb_fp_pooled_ru', 'xgb_fp_ru']


In [3]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [4]:
with open("scaffold_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=334  test=73  folds=5
Device: cpu


In [6]:
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")
y=np.load("Vectors/y.npy")

In [7]:
import importlib
import scripts.test_eval_lim_models_reg
importlib.reload(scripts.test_eval_lim_models_reg)

<module 'scripts.test_eval_lim_models_reg' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML/Section_S3/Regressor/scripts/test_eval_lim_models_reg.py'>

In [8]:
from scripts.test_eval_lim_models_reg import run_all_test_reg

test_results_reg = run_all_test_reg(
    X_fp_pooled_RU,  X_fp_pooled_poly,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_reg.json",
)

Loaded HPs from best_hp_reg.json  (21 entries)

── XGB (FP RU) ──────────────────────────────────────────

  TEST RESULTS: XGB (FP RU)
  MAE                   : 29.0817
  RMSE                  : 35.5725
  R²                    : 0.7590
  MAPE (%)              : 21.7521
  Max Error             : 91.6321
  train_time_min        : 0.0100

  Subgroup: copolymer_type
  Group                    n      MAE     RMSE       R²
  -----------------------------------------------------
  alternating             10    13.43    17.03  -0.9938
  block                    9    17.82    21.08   0.8992
  random                  23    31.29    34.38   0.7783

  Subgroup: stereo_class
  Group                    n      MAE     RMSE       R²
  -----------------------------------------------------
  achiral                  4    41.52    54.62   0.5647
  atactic                 14    37.96    44.64   0.5789
  isotactic               22    31.74    37.63   0.4798
  stereo-irregular         2    37.14    37.24 -8

In [10]:
from scripts.fig_reg_supp import plot_reg_split_validation

In [11]:
import importlib
import scripts.fig_reg_supp
importlib.reload(scripts.fig_reg_supp)

<module 'scripts.fig_reg_supp' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML/Section_S3/Regressor/scripts/fig_reg_supp.py'>

In [13]:
plot_reg_split_validation(df, trainval_idx, test_idx)

Saved: Figures/fig_reg_split_validation-scaffold.pdf
